# 01: Datasets and Preprocessing

This notebook provides a comprehensive visual exploration of the REIMS datasets. We examine class distributions, spectral signatures, and dimensionality reduction projections to understand the underlying structure of the mass spectrometry data.

In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from fishy.data.module import create_data_module


## 1. Class Distribution

The class distribution plot shows the number of samples available for each category in the dataset. A balanced dataset is generally preferred for training robust machine learning models, as it prevents the model from developing a bias toward the majority class.

In [ ]:
dm = create_data_module("species")
dm.setup()
df = dm.get_filtered_dataframe()
dist = df["Class Name"].value_counts()
px.bar(x=dist.index, y=dist.values, labels={'x': 'Class', 'y': 'Count'}, 
       title="Species Class Distribution", template="plotly_white")

## 2. Interactive Spectrum Viewer

This line plot overlays multiple individual samples from the dataset. It allows us to observe the raw variance and noise present in the REIMS spectra. By looking at individual samples, we can see if certain peaks are consistently present or if there are significant outliers.

In [ ]:
label_col = "Class Name"
feature_cols = [c for c in df.columns if c not in [label_col, "m/z"]]
sample_df = df.sample(min(10, len(df)))
melted = sample_df[[label_col]+feature_cols].melt(id_vars=label_col, var_name="m/z", value_name="Intensity")
px.line(melted, x="m/z", y="Intensity", color=label_col, 
        title="Interactive Spectrum Overlay (Random Samples)", template="plotly_white")

## 3. Mean Spectral Signatures

The mean spectral signature represents the average intensity across all samples in a class, with the shaded area indicating one standard deviation. This visualization is critical for identifying "diagnostic peaks"—specific m/z values where classes show distinct, non-overlapping behavior.

In [ ]:
X_all, y_all = dm.get_numpy_data(labels_as_indices=True)
names = dm.get_class_names()
mz_axis = np.array([float(c) for c in feature_cols])

avg_fig = go.Figure()
for idx, name in enumerate(names):
    mask = (y_all == idx)
    if np.any(mask):
        m, s = X_all[mask].mean(axis=0), X_all[mask].std(axis=0)
        avg_fig.add_trace(go.Scatter(x=mz_axis, y=m, name=name, mode="lines"))
        avg_fig.add_trace(go.Scatter(x=np.concatenate([mz_axis, mz_axis[::-1]]), 
                                     y=np.concatenate([m+s, (m-s)[::-1]]), 
                                     fill="toself", fillcolor="rgba(0,100,80,0.1)", 
                                     line=dict(color="rgba(255,255,255,0)"), 
                                     hoverinfo="skip", showlegend=False))

avg_fig.update_layout(template="plotly_white", xaxis_title="m/z", yaxis_title="Intensity", title="Mean Signatures with Std Dev")

## 4. PCA and Information Retention

Principal Component Analysis (PCA) reduces the 2000+ features into a few orthogonal components. The scatter plot shows how well the classes separate in this reduced space. The **Information Retention** plot shows the cumulative explained variance; a steep curve indicates that most of the dataset's information is captured in just a few dimensions.

In [ ]:
pca_obj = PCA(n_components=10)
X_pca = pca_obj.fit_transform(X_all)
fig_pca = px.scatter(x=X_pca[:,0], y=X_pca[:,1], color=[names[i] for i in y_all], 
                     title="PCA Projection (PC1 vs PC2)", labels={'x':'PC1', 'y':'PC2'}, template="plotly_white")
fig_pca.show()

px.area(x=range(1, 11), y=np.cumsum(pca_obj.explained_variance_ratio_), 
        labels={'x': 'Components', 'y': 'Cumulative Variance'}, 
        title="PCA: Information Retention (Explained Variance)", template="plotly_white")

## 5. t-SNE Visualization

t-Distributed Stochastic Neighbor Embedding (t-SNE) is a non-linear dimensionality reduction technique that is excellent at preserving local structures. Unlike PCA, it can capture complex, non-linear relationships, often resulting in tighter, more distinct class clusters.

In [ ]:
tsne = TSNE(n_components=2, perplexity=min(30, len(X_all)-1), random_state=42)
X_tsne = tsne.fit_transform(X_all)
px.scatter(x=X_tsne[:,0], y=X_tsne[:,1], color=[names[i] for i in y_all], 
           title="t-SNE Projection", template="plotly_white")

## 6. Global Intensity Distribution

The violin plots show the distribution of the average intensity for each sample, grouped by class. This helps identify if certain classes have systematically higher or lower total ion counts, which could be a source of bias if not properly normalized.

In [ ]:
avg_int = pd.DataFrame({"Avg Intensity": X_all.mean(axis=1), "Class": [names[i] for i in y_all]})
px.violin(avg_int, x="Class", y="Avg Intensity", color="Class", 
          box=True, points="all", title="Global Intensity Distribution per Class", template="plotly_white")